# CKPT7D: 12‑Month Baselines vs GRU+Static (Fair Comparison)

This notebook **re‑trains baselines using a 12‑month observation window** and compares them to the **12‑month GRU+static model**. This ensures an apples‑to‑apples comparison under the same data horizon.


In [1]:
# Setup
import os, sys, random
import numpy as np
import pandas as pd

sys.path.append('..')

seed = 42
random.seed(seed)
np.random.seed(seed)


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print('Torch:', torch.__version__)


Torch: 2.11.0+cpu


In [3]:
from pathlib import Path
from src.data import load_online_retail_ii, clean_data
from src.features import create_temporal_splits_multi_extended
from src.baselines import (
    evaluate_model,
    train_elasticnet,
    train_random_forest,
    train_xgboost,
    train_extra_trees,
    train_hist_gb,
    train_knn,
    train_svr,
    train_mlp,
    train_poisson,
    simple_averaging,
)


## Data + 12‑Month Temporal Splits

In [4]:
file_2009 = Path('../data/Year 2009-2010.csv')
file_2010 = Path('../data/Year 2010-2011.csv')

if not (file_2009.exists() and file_2010.exists()):
    raise FileNotFoundError('Raw dataset files missing in ../data/')

raw = load_online_retail_ii(file_2009, file_2010)
clean = clean_data(raw)

obs_months = 12
horizon_months = 3

# With 12-month history, earliest cutoff should be 2010-12-01
train_cutoffs = ['2010-12-01','2011-03-01']
val_cutoff = '2011-06-01'
test_cutoff = '2011-09-01'

train_df, val_df, test_df = create_temporal_splits_multi_extended(
    clean, train_cutoffs, val_cutoff, test_cutoff,
    obs_months=obs_months, horizon_months=horizon_months
)

exclude = {'customer_id','CustomerID','target','cutoff_date','horizon_start','horizon_end'}
feature_cols = [c for c in train_df.columns if c not in exclude]

X_train = train_df[feature_cols]
y_train = train_df['target']
X_val = val_df[feature_cols]
y_val = val_df['target']
X_test = test_df[feature_cols]
y_test = test_df['target']

print('Train/Val/Test:', train_df.shape, val_df.shape, test_df.shape)
print('Feature cols:', feature_cols)


Loading Year 2009-2010...
  Loaded 525,461 rows
Loading Year 2010-2011...
  Loaded 541,910 rows

Total rows: 1,067,371

DATA CLEANING PIPELINE
Initial rows: 1,067,371
✓ Removed missing CustomerID: 243,007 rows removed
✓ Removed cancellations: 18,744 rows removed
✓ Removed invalid Quantity/Price: 71 rows removed
✓ Converted InvoiceDate to datetime
✓ Removed duplicates: 26,124 rows removed
Final rows: 779,425 (73.0% retained)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,878
Unique invoices: 36,969


CREATING TEMPORAL SPLITS (MULTI-CUTOFF TRAIN, EXTENDED)

[1/3] Training Set (Multiple Cutoffs)

--- Train cutoff: 2010-12-01 ---

Creating window (extended):
  Observation: 2009-12-01 to 2010-12-01 (12 months)
  Horizon: 2010-12-01 to 2011-03-01 (3 months)
  Observation transactions: 386,732
  Horizon transactions: 66,364
  Customers in observation: 4,266
  Final customers with features: 4,266
  Customers with target > 0: 1,411 (33.1%)

--- Train cutoff: 2011-03-

## 12‑Month Baselines

In [5]:
results_val = {}
results_test = {}

# ElasticNet
model_en = train_elasticnet(X_train, y_train, alpha=0.1, l1_ratio=0.5)
pred_en_val = model_en.predict(X_val)
pred_en_test = model_en.predict(X_test)
results_val['ElasticNet'] = evaluate_model(y_val, pred_en_val, 'ElasticNet')
results_test['ElasticNet'] = evaluate_model(y_test, pred_en_test, 'ElasticNet')

# RandomForest
model_rf = train_random_forest(X_train, y_train, n_estimators=300, max_depth=10)
pred_rf_val = model_rf.predict(X_val)
pred_rf_test = model_rf.predict(X_test)
results_val['RandomForest'] = evaluate_model(y_val, pred_rf_val, 'RandomForest')
results_test['RandomForest'] = evaluate_model(y_test, pred_rf_test, 'RandomForest')

# XGBoost
model_xgb = train_xgboost(X_train, y_train, n_estimators=300, max_depth=5, learning_rate=0.1)
if model_xgb is not None:
    pred_xgb_val = model_xgb.predict(X_val)
    pred_xgb_test = model_xgb.predict(X_test)
    results_val['XGBoost'] = evaluate_model(y_val, pred_xgb_val, 'XGBoost')
    results_test['XGBoost'] = evaluate_model(y_test, pred_xgb_test, 'XGBoost')
else:
    pred_xgb_val = pred_xgb_test = None

# ExtraTrees
model_et = train_extra_trees(X_train, y_train, n_estimators=300, max_depth=None)
pred_et_val = model_et.predict(X_val)
pred_et_test = model_et.predict(X_test)
results_val['ExtraTrees'] = evaluate_model(y_val, pred_et_val, 'ExtraTrees')
results_test['ExtraTrees'] = evaluate_model(y_test, pred_et_test, 'ExtraTrees')

# HistGB
model_hgb = train_hist_gb(X_train, y_train, max_depth=6, learning_rate=0.1)
pred_hgb_val = model_hgb.predict(X_val)
pred_hgb_test = model_hgb.predict(X_test)
results_val['HistGB'] = evaluate_model(y_val, pred_hgb_val, 'HistGB')
results_test['HistGB'] = evaluate_model(y_test, pred_hgb_test, 'HistGB')

# SVR
model_svr = train_svr(X_train, y_train, C=10.0, epsilon=0.1)
pred_svr_val = model_svr.predict(X_val)
pred_svr_test = model_svr.predict(X_test)
results_val['SVR'] = evaluate_model(y_val, pred_svr_val, 'SVR')
results_test['SVR'] = evaluate_model(y_test, pred_svr_test, 'SVR')

# KNN
model_knn = train_knn(X_train, y_train, n_neighbors=15, weights='distance')
pred_knn_val = model_knn.predict(X_val)
pred_knn_test = model_knn.predict(X_test)
results_val['KNN'] = evaluate_model(y_val, pred_knn_val, 'KNN')
results_test['KNN'] = evaluate_model(y_test, pred_knn_test, 'KNN')

# MLP
model_mlp = train_mlp(X_train, y_train, hidden_layer_sizes=(64,32))
pred_mlp_val = model_mlp.predict(X_val)
pred_mlp_test = model_mlp.predict(X_test)
results_val['MLP'] = evaluate_model(y_val, pred_mlp_val, 'MLP')
results_test['MLP'] = evaluate_model(y_test, pred_mlp_test, 'MLP')

# Poisson
model_pr = train_poisson(X_train, y_train)
pred_pr_val = model_pr.predict(X_val)
pred_pr_test = model_pr.predict(X_test)
results_val['Poisson'] = evaluate_model(y_val, pred_pr_val, 'Poisson')
results_test['Poisson'] = evaluate_model(y_test, pred_pr_test, 'Poisson')

# SimpleAvg (EN + RF + XGB if available)
if pred_xgb_test is not None:
    pred_avg_val = simple_averaging({'ElasticNet': pred_en_val, 'RandomForest': pred_rf_val, 'XGBoost': pred_xgb_val})
    pred_avg_test = simple_averaging({'ElasticNet': pred_en_test, 'RandomForest': pred_rf_test, 'XGBoost': pred_xgb_test})
    results_val['SimpleAvg'] = evaluate_model(y_val, pred_avg_val, 'SimpleAvg')
    results_test['SimpleAvg'] = evaluate_model(y_test, pred_avg_test, 'SimpleAvg')

# Print baseline table
baseline_df = pd.DataFrame([
    {'Model': k, 'MAE': v['MAE'], 'RMSE': v['RMSE']} for k, v in results_test.items()
]).sort_values('MAE')

print('=== 12‑Month Baselines (Test) ===')
print(baseline_df.to_string(index=False))


=== 12‑Month Baselines (Test) ===
       Model      MAE     RMSE
         MLP 0.858635 1.725278
RandomForest 0.871991 1.776884
   SimpleAvg 0.873105 1.780562
  ElasticNet 0.877412 1.846906
      HistGB 0.879114 1.855712
  ExtraTrees 0.887622 1.748269
     XGBoost 0.893077 1.777009
         SVR 0.901549 2.288061
         KNN 0.923678 2.146492
     Poisson 1.207134 2.760383


## GRU + Static Features (12‑Month)
We use monthly count sequences + static features (`freq`, `monetary_total`, `latetime`).


In [6]:
# Build monthly count sequences
monthly = clean.copy()
monthly['month'] = monthly['InvoiceDate'].dt.to_period('M').dt.to_timestamp()
monthly_counts = monthly.groupby(['CustomerID', 'month']).size().unstack(fill_value=0)
monthly_counts.columns = pd.to_datetime(monthly_counts.columns)

seq_cols = [f'm{i}' for i in range(obs_months, 0, -1)]

def build_seq_features(df_split):
    seq_rows = []
    for _, row in df_split[['CustomerID', 'cutoff_date']].iterrows():
        cid = row['CustomerID']
        cutoff = pd.to_datetime(row['cutoff_date'])
        months = pd.date_range(cutoff - pd.DateOffset(months=obs_months), periods=obs_months, freq='MS')
        if cid in monthly_counts.index:
            counts = monthly_counts.loc[cid].reindex(months, fill_value=0).values
        else:
            counts = [0] * obs_months
        seq_rows.append(counts)
    return pd.DataFrame(seq_rows, columns=seq_cols, index=df_split.index)

X_train_seq = build_seq_features(train_df)
X_val_seq = build_seq_features(val_df)
X_test_seq = build_seq_features(test_df)

# Static features
static_cols = []
if 'freq' in train_df.columns:
    static_cols.append('freq')
if 'monetary_total' in train_df.columns:
    static_cols.append('monetary_total')
elif 'monetary_avg_invoice' in train_df.columns:
    static_cols.append('monetary_avg_invoice')
if 'latetime' in train_df.columns:
    static_cols.append('latetime')

X_train_static = train_df[static_cols].copy()
X_val_static = val_df[static_cols].copy()
X_test_static = test_df[static_cols].copy()

# Standardize static
stat_mean = X_train_static.mean(axis=0)
stat_std = X_train_static.std(axis=0).replace(0, 1.0)
X_train_static_n = (X_train_static - stat_mean) / stat_std
X_val_static_n = (X_val_static - stat_mean) / stat_std
X_test_static_n = (X_test_static - stat_mean) / stat_std


In [7]:
from torch.utils.data import Dataset, DataLoader

class SeqStaticDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.tensor(X_seq.values, dtype=torch.float32)
        self.X_static = torch.tensor(X_static.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    def __len__(self):
        return len(self.X_seq)
    def __getitem__(self, idx):
        return self.X_seq[idx].unsqueeze(-1), self.X_static[idx], self.y[idx]

train_ds = SeqStaticDataset(X_train_seq, X_train_static_n, y_train)
val_ds = SeqStaticDataset(X_val_seq, X_val_static_n, y_val)
test_ds = SeqStaticDataset(X_test_seq, X_test_static_n, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)


In [8]:
class GRUStaticRegressor(nn.Module):
    def __init__(self, hidden_size=32, static_dim=3, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden_size, num_layers=1, batch_first=True, dropout=0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size + static_dim, 1)
    def forward(self, x_seq, x_static):
        out, _ = self.gru(x_seq)
        last = out[:, -1, :]
        last = self.dropout(last)
        x = torch.cat([last, x_static], dim=1)
        yhat = self.fc(x)
        return F.relu(yhat).squeeze(-1)

model = GRUStaticRegressor(hidden_size=32, static_dim=len(static_cols), dropout=0.3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.L1Loss()


In [9]:
def evaluate_loader(loader):
    model.eval()
    preds = []
    trues = []
    with torch.no_grad():
        for xb, xs, yb in loader:
            yhat = model(xb, xs)
            preds.append(yhat.cpu().numpy())
            trues.append(yb.cpu().numpy())
    preds = np.concatenate(preds)
    trues = np.concatenate(trues)
    return evaluate_model(trues, preds, 'GRU_Static')

best_val = float('inf')
best_state = None
patience = 5
no_improve = 0

epochs = 30
for epoch in range(1, epochs + 1):
    model.train()
    for xb, xs, yb in train_loader:
        optimizer.zero_grad()
        yhat = model(xb, xs)
        loss = criterion(yhat, yb)
        loss.backward()
        optimizer.step()

    val_metrics = evaluate_loader(val_loader)
    val_mae = val_metrics['MAE']
    print(f"Epoch {epoch:02d} | Val MAE: {val_mae:.4f}")

    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping')
            break

if best_state is not None:
    model.load_state_dict(best_state)


Epoch 01 | Val MAE: 0.7056
Epoch 02 | Val MAE: 0.6493
Epoch 03 | Val MAE: 0.6334
Epoch 04 | Val MAE: 0.6321
Epoch 05 | Val MAE: 0.6192
Epoch 06 | Val MAE: 0.6252
Epoch 07 | Val MAE: 0.6316
Epoch 08 | Val MAE: 0.6113
Epoch 09 | Val MAE: 0.6105
Epoch 10 | Val MAE: 0.6119
Epoch 11 | Val MAE: 0.6150
Epoch 12 | Val MAE: 0.6077
Epoch 13 | Val MAE: 0.6089
Epoch 14 | Val MAE: 0.6045
Epoch 15 | Val MAE: 0.6017
Epoch 16 | Val MAE: 0.6009
Epoch 17 | Val MAE: 0.6035
Epoch 18 | Val MAE: 0.6078
Epoch 19 | Val MAE: 0.5998
Epoch 20 | Val MAE: 0.5979
Epoch 21 | Val MAE: 0.6037
Epoch 22 | Val MAE: 0.5984
Epoch 23 | Val MAE: 0.6034
Epoch 24 | Val MAE: 0.5987
Epoch 25 | Val MAE: 0.6008
Early stopping


In [10]:
gru_metrics = evaluate_loader(test_loader)
print('GRU+Static (12‑mo) Test:', gru_metrics)


GRU+Static (12‑mo) Test: {'Model': 'GRU_Static', 'MAE': 0.9283839464187622, 'RMSE': np.float64(2.242390586880086)}


## Final Comparison

In [11]:
rows = baseline_df.to_dict('records')
rows.append({'Model': 'GRU_Static_12mo', 'MAE': gru_metrics['MAE'], 'RMSE': gru_metrics['RMSE']})

compare_df = pd.DataFrame(rows).sort_values('MAE')
print(compare_df.to_string(index=False))

best_base = baseline_df.iloc[0]
print(f"Best 12‑mo baseline: {best_base['Model']} MAE={best_base['MAE']:.6f}")
print(f"GRU+Static delta: {gru_metrics['MAE'] - best_base['MAE']:+.6f}")


          Model      MAE     RMSE
            MLP 0.858635 1.725278
   RandomForest 0.871991 1.776884
      SimpleAvg 0.873105 1.780562
     ElasticNet 0.877412 1.846906
         HistGB 0.879114 1.855712
     ExtraTrees 0.887622 1.748269
        XGBoost 0.893077 1.777009
            SVR 0.901549 2.288061
            KNN 0.923678 2.146492
GRU_Static_12mo 0.928384 2.242391
        Poisson 1.207134 2.760383
Best 12‑mo baseline: MLP MAE=0.858635
GRU+Static delta: +0.069749
